In [1]:
# Get raw data
with open('input/17.txt', 'r') as f:
    rawinput = f.read().strip()

In [2]:
from intcode import Program  # From Day 9
import numpy as np

In [3]:
# Part 1
class Interface(Program):
    def __init__(self, instr, inputs=None, mode=None, verbose=False):
        if mode:
            instr[0] = mode
        super().__init__(instr, inputs or [], verbose)

interface = Interface([int(i) for i in rawinput.split(',')])
interface.do_exec()

scview = np.array([list(i) for i in ''.join([chr(i) for i in interface.output]).strip().split('\n')])

isect_filt = np.all(np.stack([z[i+1:, j+1:][:scview.shape[0], :scview.shape[1]]
                              for z in [np.pad(scview, 1, constant_values='.')]
                              for i in [-1,0,1] for j in [-1,0,1] if 0 in [i,j]],
                             axis=-1) != '.',
                    axis=2).reshape(-1)
np.sum(np.prod((yx:=np.column_stack((np.repeat(np.arange(scview.shape[0]), 
                                               scview.shape[1]),
                                     np.tile(np.arange(scview.shape[1]), 
                                             scview.shape[0]))))[isect_filt],
               axis=1)).item()

4800

In [4]:
# Part 2
scview_p = np.pad(scview, 1, constant_values='.')
posy, posx = yx[((scview != '.')&(scview != '#')).reshape(-1)][0]+1
raw_instr = []
while np.any((z:=scview_p[posy, posx-1::2][:2])=='#'):
    turn = np.argmin(z=='#')
    posx += (s:=np.argmax(scview_p[posy, posx::(z:=[1,-1][turn])]=='.')-1)*z
    raw_instr += [f"{'RL'[turn]}{s.item()}"]
    scview_p, posy, posx = (np.flip(scview_p.T, axis=turn),
                            [scview_p.shape[1-turn]-posx-1, posx][turn], 
                            [posy, scview_p.shape[1-turn]-posy-1][turn])
raw_instr = ''.join(raw_instr)

def get_move_instr(instr, nf=None):
    nf = nf or 'A'
    if {*instr}&{*'RL'}:
        return sorted([[i+len(z:=u.replace('L',',L,').replace('R',',R,')[1:]), [z]+j] 
                       for u in {j[:n]
                                 for j in [''.join([[i,' '][64<ord(i)<76] for i in instr]).split()[0]]
                                 for m in [[k for k,l in enumerate(j) if l in 'LR'][1:]+[len(j)]]
                                 for n in m}
                       for i,j in [get_move_instr(nf.join(instr.split(u)), 
                                                  chr(ord(nf)+1))]])[0]
    else:
        return [len(z:=','.join(instr)), [z]]
    
move_instr = [(z:=get_move_instr(raw_instr)[1])[-1]] + z[:-1] + ['n']
ascii_instr = [ord(j) for j in ''.join([f'{i}\n' for i in move_instr])]

interface = Interface([int(i) for i in rawinput.split(',')],
                      inputs=ascii_instr,
                      mode=2,
                      verbose=False)
interface.do_exec()
interface.output[-1]

982279